In [ ]:
!git clone https://github.com/cszn/SCUNet

In [25]:
%cd ./SCUNet

!pip install -q timm einops thop tifffile scikit-image
!python3 main_download_pretrained_models.py --models "SCUNet" --model_dir "model_zoo"

LICENSE
README.md
cog.yaml
figs
main_download_pretrained_models.py
main_test_scunet_color_gaussian.py
main_test_scunet_gray_gaussian.py
main_test_scunet_real_application.py
model_zoo
models
predict.py
results
testsets
utils


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

from models.network_scunet import SCUNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 2
LR = 1e-4
SAVE_PATH = "../ct_scunet_finetuned.pth"


def evaluate(
    model: torch.nn.Module,
    loader: DataLoader,
) -> tuple[float, float, float]:
    model.eval()

    losses: list[float] = []
    psnrs: list[float] = []
    ssims: list[float] = []

    with torch.no_grad():
        for noisy, clean in loader:
            noisy: torch.Tensor = noisy.to(DEVICE)
            clean: torch.Tensor = clean.to(DEVICE)

            pred: torch.Tensor = model(noisy).clamp(0, 1)

            loss: float = F.l1_loss(pred, clean).item()
            losses.append(loss)

            for i in range(pred.shape[0]):
                pred_np: np.ndarray = pred[i, 0].cpu().numpy()
                clean_np: np.ndarray = clean[i, 0].cpu().numpy()

                psnrs.append(
                    peak_signal_noise_ratio(clean_np, pred_np, data_range=1.0)
                )
                ssims.append(
                    structural_similarity(clean_np, pred_np, data_range=1.0)
                )

    model.train()
    return float(np.mean(losses)), float(np.mean(psnrs)), float(np.mean(ssims))

In [ ]:
quick_loader = DataLoader(
    Subset(test_ds, range(10)),
    batch_size=1,
    shuffle=False,
    num_workers=1,
)

checkpoints_to_test = [
    "scunet_gray_15.pth",
    "scunet_gray_25.pth",
    "scunet_gray_50.pth",
]

results: dict[str, tuple[float, float, float]] = {}

for ckpt_name in checkpoints_to_test:
    test_model = SCUNet(in_nc=1, config=[4, 4, 4, 4, 4, 4, 4], dim=64)

    ckpt = torch.load(f"model_zoo/{ckpt_name}", map_location="cpu")
    test_model.load_state_dict(ckpt, strict=True)
    test_model = test_model.to(DEVICE)
    test_model.eval()

    loss, psnr, ssim = evaluate(test_model, quick_loader)
    results[ckpt_name] = (loss, psnr, ssim)

    print(f"{ckpt_name:25s} | l1={loss:.5f} | psnr={psnr:.2f} | ssim={ssim:.4f}")

    del test_model
    torch.cuda.empty_cache()

best_ckpt = max(results, key=lambda k: results[k][1])  # по PSNR
print(f"\nЛучший стартовый чекпоинт по PSNR: {best_ckpt}")

In [ ]:
model = SCUNet(in_nc=1, config=[4, 4, 4, 4, 4, 4, 4], dim=64)
ckpt = torch.load(f"model_zoo/{best_ckpt}", map_location="cpu")
model.load_state_dict(ckpt, strict=True)
model = model.to(DEVICE)

optimizer: torch.optim.Optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
)

best_psnr: float = -1.0

for epoch in range(EPOCHS):
    model.train()
    train_loss: float = 0.0

    for noisy, clean in train_loader:
        noisy: torch.Tensor = noisy.to(DEVICE)
        clean: torch.Tensor = clean.to(DEVICE)

        pred: torch.Tensor = model(noisy)
        loss: torch.Tensor = F.l1_loss(pred, clean)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss, val_psnr, val_ssim = evaluate(model, val_loader)

    print(
        f"epoch {epoch+1}/{EPOCHS} | "
        f"train_l1={train_loss:.5f} | "
        f"val_l1={val_loss:.5f} | "
        f"val_psnr={val_psnr:.2f} | "
        f"val_ssim={val_ssim:.4f}"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(model.state_dict(), SAVE_PATH)
        print("saved best:", SAVE_PATH)